In [59]:
import math
import re
from collections import defaultdict


def tokenize(message):
    message = message.lower()
    all_words = re.findall("[a-z0-9']+", message)
    return set(all_words)

def count_words(training_set):
    """ (message, is_spam) """
    counts = defaultdict(lambda: [0, 0])
    for message, is_spam in training_set:
        for word in tokenize(message):
            counts[word][0 if is_spam else 1] += 1
    return counts

def word_probabilities(counts, total_spams, total_non_spams, k=0.5):
    return [(w, (spam + k) / (total_spams + 2 * k), (non_spam + k) / (total_non_spams + 2 * k))
            for w, (spam, non_spam) in counts.items()]

def spam_probability(word_probs, message):
    message_words = tokenize(message)
    log_prob_if_spam = log_prob_if_not_spam = 0.0
    for word, prob_if_spam, prob_if_not_spam in word_probs:
        if word in message_words:
            log_prob_if_spam += math.log(prob_if_spam)
            log_prob_if_not_spam += math.log(prob_if_not_spam)
        else:
            log_prob_if_spam += math.log(1.0 - prob_if_spam)
            log_prob_if_not_spam += math.log(1.0 - prob_if_not_spam)
    prob_if_spam = math.exp(log_prob_if_spam)
    prob_if_not_spam = math.exp(log_prob_if_not_spam)
    return 1 if prob_if_spam / (prob_if_spam + prob_if_not_spam) > 0.50 else 0

In [60]:
class NaiveBayesClassifier:
    def __init__(self, k=0.5):
        self.k = k
        self.word_probs = []

    def train(self, training_set):
        num_spams = len([is_spam for message, is_spam in training_set if is_spam])
        num_non_spams = len(training_set) - num_spams
        word_counts = count_words(training_set)
        self.word_probs = word_probabilities(word_counts, num_spams, num_non_spams, self.k)

    def classify(self, message):
        return spam_probability(self.word_probs, message)

In [61]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv("../data/spam.csv", encoding='latin-1')
data = data[["v1", "v2"]]
data["v1"] = data["v1"].map({"ham": 0, "spam":1})

train_data, test_data = train_test_split(data, random_state=42, test_size=0.2)
train_data = list(zip(train_data['v2'],train_data['v1']))
test_data = list(zip(test_data['v2'], test_data['v1']))

model = NaiveBayesClassifier()
model.train(train_data)

In [73]:
count = 0
true_positives = 0
true_negatives = 0
false_positives = 0
false_negatives = 0

for message_test, is_spam_test in test_data:
    if model.classify(message_test) == is_spam_test:
        count += 1
        if is_spam_test == 1: true_positives += 1
        else: true_negatives += 1
    else:
        if is_spam_test == 1: false_negatives += 1
        else: false_positives += 1
print(f"Accuracy: {count/len(test_data):.3f}\n"
      f"True positives: {true_positives}\n"
      f"True negatives: {true_negatives}\n"
      f"False positives: {false_positives}\n"
      f"False negatives: {false_negatives}\n")

Accuracy: 0.984
True positives: 133
True negatives: 964
False positives: 1
False negatives: 17

